# 🎯 [D-8] 파이프라인 검증용 드라이 런 (Dry Run) 테스트

이 노트북은 실제 400GB 데이터셋이 없는 상황에서, 우리가 작성한 `Swin-YOLO26` 커스텀 모델 구조와 `train.py` 학습 파이프라인이 에러 없이(특히 텐서 차원 충돌이나 OOM 없이) 정상 동작하는지 검증하기 위한 **더미 데이터(Dummy Data) 테스트 환경**입니다.

### 💡 테스트 목표
1. 커스텀 파이썬 모델(`.py`) 동적 임포트 정상 작동 여부 확인
2. 모델 순전파(Forward) 시 백본-넥 차원 정렬(Shape Match) 검증
3. 단 1 에폭(Epoch) 학습을 통한 학습 파이프라인 관통 테스트

In [1]:
# 🛠️ Step 1. 5초 만에 만드는 '더미 데이터셋' 자동 생성기
import os
import numpy as np
from PIL import Image

def create_dummy_dataset():
    # 현재 노트북 위치(notebooks/)를 기준으로 프로젝트 루트 계산
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    base_dir = os.path.join(PROJECT_ROOT, 'dummy_data')
    img_dir = os.path.join(base_dir, 'images')
    lbl_dir = os.path.join(base_dir, 'labels')
    
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    # 1. 가짜 이미지 4장 및 라벨 생성 (640x640 노이즈 이미지)
    img_paths = []
    for i in range(4):
        img_array = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        img_path = os.path.join(img_dir, f'dummy_{i}.jpg')
        img.save(img_path)
        img_paths.append(img_path)

        # 가짜 YOLO 라벨 (클래스 0, x_center, y_center, width, height)
        lbl_path = os.path.join(lbl_dir, f'dummy_{i}.txt')
        with open(lbl_path, 'w') as f:
            f.write("0 0.5 0.5 0.2 0.2\n")

    # 2. train_list.txt 생성
    list_path = os.path.join(base_dir, 'train_list.txt')
    with open(list_path, 'w') as f:
        for p in img_paths:
            f.write(f"{p}\n")

    # 3. dummy.yaml 명세서 생성 (YOLO가 읽을 데이터 경로)
    # 🚨 경로 꼬임을 막기 위해 절대 경로로 기록합니다.
    yaml_path = os.path.join(base_dir, 'dummy.yaml')
    with open(yaml_path, 'w', encoding='utf-8') as f:
        f.write(f"train: {list_path}\n")
        f.write(f"val: {list_path}\n") 
        f.write(f"nc: 7\n")
        f.write(f"names: ['워터스포팅', '흐름', '도막분리', '핀홀', '균열', '부풀음', '이물질포함']\n")

    print(f"✅ 더미 데이터셋 생성 완료!\n저장 경로: {yaml_path}")

# 실행
create_dummy_dataset()

✅ 더미 데이터셋 생성 완료!
저장 경로: c:\Users\trust\Desktop\swin-yolo26\swin-yolo26-paint-defect\dummy_data\dummy.yaml


In [2]:
# ⚙️ Step 2. 드라이 런 전용 테스트 설정 파일(Config) 생성
# 주피터의 매직 커맨드(%%writefile)를 사용하여 configs 폴더에 파일을 직접 씁니다.

CONFIG_PATH = "../configs/model_test.yaml"

In [5]:
%%writefile ../configs/model_test.yaml
# ==============================================================
# 파이프라인 검증용 Dry Run 설정 파일 (1 Epoch 테스트용)
# ==============================================================
# 커스텀 아키텍처 연동 테스트 (swin_yolo26.py 작성 전이라면 "yolo11m.pt"로 테스트)
model_path: "yolo11m.pt"    # 🚨 나중에 "../models/swin_yolo26.py"로 변경하여 테스트하세요!

# 더미 데이터 경로 (노트북에서 생성한 절대 경로를 사용하므로 train.py에서 처리됨)
data: "../dummy_data/dummy.yaml"

device: cpu                 # Intel Ultra 7
batch: 2                    # 더미 이미지가 4장뿐이므로 배치 2
workers: 8                  # 원격 데스크톱의 "RTX 3090 서버"로 코드를 옮겨서 돌릴 때
imgsz: 640
epochs: 1                   # 단 1 에폭만 돌려서 파이프라인 완주 여부만 확인
val: False                  # 더미 데이터이므로 검증은 생략
plots: False

Overwriting ../configs/model_test.yaml


### 🚨 Step 3. `train.py` 임시 수정 (매우 중요)
현재 우리가 작성해둔 `scripts/train.py` 코드를 보면, 실제 학습을 위해 데이터 경로를 강제 할당하는 부분이 있습니다. 더미 테스트를 위해 이 부분을 **주석 처리하거나 임시 변경**해 주어야 합니다.

`scripts/train.py` 파일을 열고 아래 부분을 찾아주세요.
```python
# 🚨 데이터셋 명세서의 절대 경로를 강제 할당합니다.
# YAML_PATH = os.path.join(PROJECT_ROOT, "data", "ship_paint_data.yaml") # 원본 (잠시 주석 처리)

# 테스트용 더미 데이터 경로로 임시 교체 💡
YAML_PATH = os.path.join(PROJECT_ROOT, "dummy_data", "dummy.yaml")

In [6]:
# 🚀 Step 4. 드라이 런 실행!
# 터미널 명령어(!)를 사용하여 우리가 만든 설정 파일로 학습을 시작합니다.
# 단 1 Epoch이므로 수십 초 내로 끝나야 정상입니다.

!python ../scripts/train.py --config ../configs/model_test.yaml

🚀 읽어들인 설정 파일: c:\Users\trust\Desktop\swin-yolo26\swin-yolo26-paint-defect\configs\model_test.yaml
🧠 로드할 모델 가중치: yolo11m.pt
📊 강제 할당된 데이터 경로: c:\Users\trust\Desktop\swin-yolo26\swin-yolo26-paint-defect\dummy_data\dummy.yaml
📁 결과물 저장 폴더: c:\Users\trust\Desktop\swin-yolo26\swin-yolo26-paint-defect\runs/exp_model_test
--------------------------------------------------
New https://pypi.org/project/ultralytics/8.4.57 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.56  Python-3.10.20 torch-2.5.1+cu121 CPU (Intel Core Ultra 7 165U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\trust\Desktop\swin-yolo26\swin-yolo26-paint-defect\dummy_data\dummy.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=